# Building generative models

The [`GenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel) is a very central object to the codebase and is responsible for:
- Defining the loss function via `get_loss`. This method is called in [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html#loss-pipeline) which orchestrates training steps.
    > ***We will talk a lot about how the choice of this specific propagates into many other methods. The loss completely determines what the network predicts, and as such, it totally determines how to derive sample-time quantities. Thus, at the point of specifying the loss, the user must also specify the conversion methods below.***
- Providing methods for sample-time functions:
    - `get_generator`, **always required**. It returns a per-modality [`Generator`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.Generator) — for instance a `Velocity` (for ODE-only sampling) or a `VelocityAndScore` (whose score is needed for SDE sampling), with the concrete type set per modality via the interpolant's inferred `generator_type`. Every sampler calls it.
    - `get_guidance_loss`, **optional**. A loss used to compute a surrogate conditional score for classifier-style intrinsic conditional guidance recipes.
- It also holds the important methods
    - `get_network_output`, merely calls the network
    - `get_embeddings`, embeds all modalities using their respective [`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.Embedder)s.

> ***You will notice that the `GenerativeModel` has, as attributes, all the components which contain learnable parameters.*** This is intentional design so that gradients can propagate through correctly.

In the [training and sampling tutorial](./1.training_and_sampling.ipynb) we glossed over the generative model in a single cell by using the concrete implementation [`VelocityOneSidedGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.factory.VelocityOneSidedGenerativeModel). This is great for beginners to `stix`, but many users will want to construct their own. In this tutorial we will walk through how one can do this.


## Structure of the notebook

1. Imports
2. Dataloader, [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html#stix.core.modality.ModalityRegistry), and [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network)
    1. Dataloader
    2. `ModalityRegistry`
    3. `Network`
3. `GenerativeModel`
    1. Simple denoiser model
    2. Simple noise prediction model
    3. Advanced hybrid (velocity and logits) model


## 0. Installation

We recommend running this notebook in a **fresh virtual environment**. 

Copy the notebook into some new directory. Then, from a terminal, in the new directory containing the notebook (`3.generative_model.ipynb`):
```
python -m venv my_env
source my_env/bin/activate
pip install notebook ipykernel

python -m ipykernel install --user --name my_env --display-name "my_env"

jupyter notebook
```

The next cell installs `stix` from PyPI.

In [ ]:
%pip install stix-ml

## 1. Imports

We only need a slice of `stix` here.

In [ ]:
import logging
from functools import partial

import grain
import jax
import jax.numpy as jnp
import jax.random as jr
from flax import nnx
from jaxtyping import PyTree

# Embedders bridge raw data space and the network's embedding space.
from stix.core.embedder import IdentityEmbedder, OneHotDiscreteEmbedder

# The abstract generative-model base; the models below subclass it directly.
from stix.core.gen_model import GenerativeModel
from stix.core.generator import VelocityAndScore

# Pre-baked interpolant recipe (picks beta_t, gamma_t).
from stix.core.interpolant import FlowMatchingOneSidedInterpolant

# Off-the-shelf criteria
from stix.core.loss.criterion import CrossEntropyCriterion, MSECriterion

# Helper function to reduce pytrees of loss scalars to a single scalar.
from stix.core.loss.utils import reduce_modality_losses

# The modality registry: single source of truth for per-modality config.
from stix.core.modality import ModalityRegistry

# The abstract network contract — we subclass it with a tiny inline MLP below.
from stix.nn import Network

# Core data types for a training batch.
from stix.typing import Batch, RawSourceTargetPair

In [ ]:
stix_logger = logging.getLogger("stix")
stix_logger.setLevel(logging.INFO)

key = jax.random.PRNGKey(0)

## 2. Dataloader, [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html#stix.core.modality.ModalityRegistry), [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.Network)

These are components which are needed to initialise a `GenerativeModel`. The constructor for `GenerativeModel` is very simple, we require:

- `ModalityRegistry`: this object is covered in [1.training_and_sampling](./1.training_and_sampling.ipynb). A few key details
    - It is the one-source-of-truth for the [PyTree](https://docs.jax.dev/en/latest/pytrees.html) structure of the [`Modality`](https://instadeepai.github.io/stix/api_reference/core/modality.html#stix.core.modality.Modality)s.
    - It is initialised from a batch of data so it can infer the correct structure. It also extracts the shape and whether the modality is discrete, `is_discrete`, for each `Modality`.
    - It contains, at minimum, the [`Embedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.Embedder) and the [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.Interpolant) for each `Modality`.
- `Network`: the model which is used to create predictions.


We will go through these elements very quickly. Again, see [1.training_and_sampling](./1.training_and_sampling.ipynb) for more information.

### 2.1 Dataloader
**We are going to skip over many details** as understanding the data is not important for this tutorial. See [2.grain_multimodal_dataloading](./2.grain_multimodal_dataloading.ipynb) for in-depth advice on how to set up a dataset in `stix`, and how to use grain.

In this tutorial, we will be using a very minimal function that samples two data modalities from a Gaussian mixture model (GMM):
- `coordinates` — 2D coordinates of the sampled data point in the $(x, y)$ plane.
- `index` — The index corresponding to the mixture component the coordinates were sampled from.

In [ ]:
BATCH_SIZE = 256

CORNERS = jnp.array(
    [[-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0], [1.0, 1.0]], dtype=jnp.float32
)


@jax.jit  # Note: one compiled call builds the whole batch! See the pipeline below
def sample_gmm(batch_indices: jax.Array, key: jax.Array) -> Batch:
    """Draw one GMM sample per index: pick a corner, scatter Gaussian noise around it.

    `grain` shuffles and groups the indices; we vectorise the draw over them with
    ``jax.vmap``, giving each index its own key via ``jax.random.fold_in`` so every
    sample is reproducible. Returns the 2D coordinates and their one-hot corner
    indices wrapped in a single batched ``Batch``.
    """

    def draw_one(sample_key: jax.Array) -> tuple[jax.Array, jax.Array]:
        """One sample: the maths stays per-sample, ``vmap`` turns it into a batch."""
        idx_key, noise_key = jax.random.split(sample_key)
        corner = jax.random.randint(idx_key, (), 0, len(CORNERS))  # which mode
        coordinates = (
            jax.random.normal(noise_key, (2,), jnp.float32) * 0.2 + CORNERS[corner]
        )
        return coordinates, corner

    keys = jax.vmap(partial(jax.random.fold_in, key))(batch_indices)
    coordinates, corner = jax.vmap(draw_one)(keys)

    raw_batch = {
        "coordinates": RawSourceTargetPair(source=None, target=coordinates),
        "index": RawSourceTargetPair(
            source=None, target=jax.nn.one_hot(corner, len(CORNERS))
        ),
    }
    return Batch(raw_batch=raw_batch, is_discrete={"coordinates": False, "index": True})


def gmm_dataset(seed: int):
    """Helper function to instantiate a grain dataset."""
    return (
        grain.MapDataset.range(int(1e9))
        .seed(seed)
        .shuffle()
        .repeat()
        .batch(BATCH_SIZE, drop_remainder=True)
        .map(partial(sample_gmm, key=jr.key(seed)))
        .to_iter_dataset()
    )


train_iter = iter(gmm_dataset(seed=0))
validation_iter = iter(gmm_dataset(seed=42))

### 2.2  [`ModalityRegistry`](https://instadeepai.github.io/stix/api_reference/core/modality.html#stix.core.modality.ModalityRegistry)

For a reminder of how this object works, we refer the user to [1.training_and_sampling](./1.training_and_sampling.ipynb).

In [ ]:
# Build the registry skeleton from a batch: `from_batch` reads each modality's
# shape and discreteness. The remaining fields are filled below.
batch = next(train_iter)
modality_registry = ModalityRegistry.from_batch(batch)

# Interpolant: the canonical flow-matching schedule (beta_t = t, gamma_t = 1 - t),
# shared across every modality.
one_sided_interpolant = FlowMatchingOneSidedInterpolant()
modality_registry.set("interpolant", one_sided_interpolant)

# Embedders (shape-dependent, so installed with per-modality factories): a
# one-hot embedder for discrete modalities, the identity embedder for continuous ones.
modality_registry.set(
    field="embedder",
    value=lambda modality: (
        OneHotDiscreteEmbedder(dm_shape=modality.shape)
        if modality.is_discrete
        else IdentityEmbedder(dm_shape=modality.shape)
    ),
    is_factory=True,
    use_deepcopy=True,
)

# Generator type: at sampling time each modality yields a `VelocityAndScore`
# generator (a velocity for the ODE and a score for the SDE).

### 2.3 [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#network)

Any network plugged into a `GenerativeModel` need only satisfy the [`Network`](https://instadeepai.github.io/stix/api_reference/nn/network.html#network) contract: it is called as `network(z_t, t, context_data, context_mask, attention_mask)` and returns a per-modality output pytree.

Here we build a small **cross-modal** MLP so the modality representations interact in the network. To achieve this we concatenate every modality (and time) into one vector and feed this into a shared trunk (the "backbone"), then route the fused state back through one output head per modality. This is the classic *unified backbone + modality-specific heads* pattern.
> **Note**: This is flat version of `stix`'s very own [`EncoderBackboneDecoderNetwork`](https://instadeepai.github.io/stix/api_reference/nn/network.html#stix.nn.EncoderBackboneDecoderNetwork).

In [ ]:
def _is_module_leaf(leaf):
    """Stop pytree traversal at nnx.Module boundaries.

    The per-modality heads are a pytree whose leaves are the head modules;
    without this, `jax.tree.*` would descend into each head's parameters.
    """
    return isinstance(leaf, nnx.Module)


class CrossModalMLPNetwork(Network):
    """Cross-modal MLP: concatenate every modality, fuse in a shared trunk, then
    route the fused state back through one head per modality.

    Unlike a per-modality network, every output depends on *every* input: the
    shared trunk sees all modalities (and time) at once, so `coordinates` and
    `index` can inform each other. This is the classic "unified backbone +
    modality-specific heads" pattern — a flat miniature of stix's own
    `EncoderBackboneDecoderNetwork`.
    """

    def __init__(self, embedding_dims: PyTree[int], hidden_dim: int, rngs: nnx.Rngs):
        """Build a shared trunk over the concatenated modalities, plus one output head per modality."""
        # Shared trunk (the "backbone"): fuses all modalities + time into one
        # representation. This concatenation is where the modalities interact.
        fused_input_dim = sum(jax.tree.leaves(embedding_dims)) + 1  # +1 for time
        self.trunk = nnx.Sequential(
            nnx.Linear(fused_input_dim, hidden_dim, rngs=rngs),
            nnx.silu,
            nnx.Linear(hidden_dim, hidden_dim, rngs=rngs),
            nnx.silu,
        )

        # Per-modality output heads, held as a pytree (nnx.data, not a keyed dict)
        # so they mirror the modality structure and stay trainable under nnx. Each
        # projects the fused representation back to its modality's own dimension.
        self.heads = nnx.data(
            jax.tree.map(
                lambda dim: nnx.Linear(hidden_dim, dim, rngs=rngs), embedding_dims
            )
        )

    def __call__(
        self, z_t, t, context_data=None, context_mask=None, attention_mask=None
    ):
        """Concatenate the modalities (+ time), fuse in the trunk, then route back per modality."""
        # 1. Concatenate every modality (deterministic treedef order) plus a time
        # column into one vector.
        modalities = jax.tree.leaves(z_t)
        batch_shape = modalities[0].shape[:-1]
        t_col = jnp.broadcast_to(jnp.atleast_1d(t), batch_shape + (1,))
        fused_input = jnp.concatenate(modalities + [t_col], axis=-1)

        # 2. Fuse in the shared trunk -> every output now sees every modality.
        fused = self.trunk(fused_input)

        # 3. Route the shared representation back through each modality's head,
        # rebuilding the input pytree structure.
        return jax.tree.map(
            lambda head: head(fused), self.heads, is_leaf=_is_module_leaf
        )


key, model_key = jr.split(key)  # network initialisation
rngs = nnx.Rngs(model_key)

# One output head per modality, sized off the registry's embedding dims.
network = CrossModalMLPNetwork(
    embedding_dims=modality_registry.map(lambda m: m.embedder.embedding_shape[-1]),
    hidden_dim=128,
    rngs=rngs,
)

## 3. Generative Model
To build a generative model we must subclass `GenerativeModel`. As a minimum we must provide implementations for `get_loss` and `get_generator`.

Let's give a quick reminder from earlier. `get_loss`, defines the loss which in turn is eventually called in [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html#loss-pipeline) which orchestrates the Monte Carlo approximation of the loss. In turn this defines what the network actually predicts. Once you know what the network predicts then it is possible to write `get_generator`, which converts whatever it is the network predicted into the sample-time [`Generator`](https://instadeepai.github.io/stix/api_reference/core/generator.html#stix.core.generator.Generator) for each modality. The concrete generator type is set per modality via the interpolant's inferred `generator_type`: a `Velocity` carries just the velocity used by both ODE and SDE solvers, while a `VelocityAndScore` additionally carries the score that SDE solvers require.

There is one additional, optional method you may want to override: [`get_guidance_loss`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.GenerativeModel.get_guidance_loss) (the base implementation raises `NotImplementedError`). This turns the model's own network output into a scalar loss against conditioning data, and is the sole hook consumed by [`get_intrinsic_guidance_generator`](https://instadeepai.github.io/stix/api_reference/sampling/guidance.html#stix.sampling.guidance.get_intrinsic_guidance_generator), a recipe for classifier-style guidance/conditional sampling. There is no default: the right loss depends on what you condition on as well as on what the network predicts, so you write it yourself — typically one criterion per modality (an MSE against the clean prediction for continuous modalities, a cross-entropy on logits for discrete ones), summed over the registry. See [4.conditioning_and_guidance.ipynb](./4.conditioning_and_guidance.ipynb) for a worked implementation. 

> The velocity and score carried by the generator returned from `get_generator`, and the predictions inside `get_guidance_loss`, are all merely mathematical conversions of the network output, `network_output`. E.g. if the network has been trained to predict velocity then the generator's velocity will just be `network_output`, but e.g. its score will be some function which manipulates your specific [`Interpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.Interpolant) $z_t = I_t(z_{\mathrm{src}}, z_{\mathrm{tgt}},\epsilon)$. Fortunately, as we will go on to see, for common interpolant families, namely [`OneSidedLinearStochasticInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.OneSidedLinearStochasticInterpolant), whose form is $z_t = \beta_t z_{\mathrm{tgt}} + \gamma_t \epsilon$, we have provided many of these conversions for you.

## 3.1 Simple denoiser model
Let's begin with a simple denoiser generative model, i.e. a model that is trained to predict the denoised variables $z_{\mathrm{tgt}}$. To simplify things, we make the following assumptions on the continuous and discrete data modalities we are considering:

- Discrete `Modality` (a `Modality` such that `modality.is_discrete=True`): we suppose the raw discrete variables are integer indices, which are embedded using a [`OneHotDiscreteEmbedder`](https://instadeepai.github.io/stix/api_reference/core/embedder.html#stix.core.embedder.OneHotDiscreteEmbedder). The embedded target variables $z_{\mathrm{tgt}}$ thus live on the corners of simplex $\mathcal{S}^{K-1} = \{z \in [0,1]^{K}: \sum_{i=1}^{K} z_i = 1\}$. 
   > In this tutorial, the embedded variables corresponding to discrete modalities are treated as *continuous* quantities, and use a `ContinuousInterpolant`. See [tutorial 6. on discrete models](./6.discrete_models.ipynb) for examples of generative models using `DiscreteInterpolant`.


- Continuous `Modality` (a `Modality` such that `modality.is_discrete=False`): we suppose these modalities do not require an embedding and use an `IdentityEmbedder`.

Remember our interpolants, as defined in the `ModalityRegistry` (in Section 2.2), are simple `FlowMatchingOneSidedInterpolant`s, each a specific instance of a `OneSidedLinearStochasticInterpolant`,

\begin{equation}
\begin{aligned}
\mathrm{One\ sided\ linear\ stochastic\ interpolant}:\quad\quad & z_t = \beta_t z_{\mathrm{tgt}} + \gamma_t \epsilon \\
\mathrm{Flow\ matching\ one\ sided\ interpolant}:\quad\quad & z_t = t z_{\mathrm{tgt}} + (1-t) \epsilon
\end{aligned}
\end{equation}

To create our simple denoiser model, we follow these steps:

Step 1:
**Loss**: Create an MSE loss between the ground truth denoised target variable $z_{\mathrm{tgt}}$ and our network output. In the last section of this tutorial, we will give an example of a generative model treating the network output as logits and using a cross-entropy loss for discrete modalities.

Step 2:
**Velocity**: Denoting velocity as $v_t$. We need to convert $z_{\mathrm{tgt}}$ to $v_t$, well $v_t = \dot{z}_t = z_{\mathrm{tgt}} - \epsilon$. Then, we know that $\epsilon = \frac{z_t - t z_{\mathrm{tgt}}}{1-t}$, thus $v_t = z_{\mathrm{tgt}} - \frac{z_t - tz_{\mathrm{tgt}}}{1-t} = \frac{z_{\mathrm{tgt}} - z_t}{1-t}$.

Step 3:
**Score**: Firstly, notice that the score is well defined because we have a [`ContinuousStochasticInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.ContinuousStochasticInterpolant) with $\gamma_t>0$ for $t\in(0,1)$. Next, let's say we **are** interested in SDE sampling. 

Denoting the score as $s_t$. We know that $s_t = -\frac{\epsilon}{\gamma_t} = - \frac{\epsilon}{1-t}$. We also have $z_t = t z_{\mathrm{tgt}} + (1-t) \epsilon$. Thus $s_t =  \frac{t z_{\mathrm{tgt}} - z_t}{(1-t)^2}$


In [ ]:
class DenoiserGenerativeModel(GenerativeModel):
    """Denoiser generative model."""

    def get_loss(self, net_out, t, z_t, epsilon, embedded_pairs, raw_pairs, loss_mask):
        """Compute the loss. An MSE between embedded target variables and our network output."""
        # Use one of our pre-defined criterion
        mse = MSECriterion()

        # Define the per-modality loss as a function
        def _per_modality_network_output_to_loss(
            modality, net_out_k, embedded_pairs_k, loss_mask_k
        ):
            return mse(net_out_k, embedded_pairs_k.target, t, loss_mask_k)

        # Use `.map` to apply the function over the whole pytree
        per_modality_loss = self.modality_registry.map(
            _per_modality_network_output_to_loss, embedded_pairs, loss_mask
        )

        # Reduce the pytree to a scalar by computing the mean
        return jnp.mean(jnp.stack(jax.tree.leaves(per_modality_loss)))

    def get_generator(self, net_out, z_t, t):
        r"""Convert network output to a per-modality `VelocityAndScore` generator.

        We have $v_t = \frac{z_{\mathrm{tgt}} - z_t}{1-t}$ and $s_t =  \frac{t z_{\mathrm{tgt}} - z_t}{(1-t)^2}$.
        The maths is derived in the markdown above!
        """

        # Define the per-modality conversion as a function
        def _per_modality_network_output_to_generator(modality, net_out_k, z_t_k):
            velocity = (net_out_k - z_t_k) / (1 - t)
            score = (t * net_out_k - z_t_k) / ((1 - t) ** 2)
            return VelocityAndScore(velocity=velocity, score=score)

        # Use `.map` to apply the function over the whole pytree
        return self.modality_registry.map(
            _per_modality_network_output_to_generator, net_out, z_t
        )

In [ ]:
denoiser_gen_model = DenoiserGenerativeModel(network, modality_registry)

As we mentioned, for [`OneSidedLinearStochasticInterpolant`](https://instadeepai.github.io/stix/api_reference/core/interpolant.html#stix.core.interpolant.OneSidedLinearStochasticInterpolant) we actually provide generic conversion methods for you. 

Below we demonstrate how these converters can be accessed via each modality's interpolant, e.g. `modality.interpolant.velocity_from_target(...)`. For a comprehensive list of all the converters available, check out the [`Interpolant` documentation](https://instadeepai.github.io/stix/api_reference/core/interpolant.html).

We will subclass our newly defined `DenoiserGenerativeModel` and overwrite `get_generator` using these pre-defined convertors. 

In [ ]:
class ConverterDenoiserGenerativeModel(DenoiserGenerativeModel):
    """Denoiser generative model using pre-baked converters."""

    def get_generator(self, net_out, z_t, t):
        r"""Convert network output to a per-modality `VelocityAndScore` generator.

        Use the `OneSidedLinearStochasticInterpolant` conversion methods
        `.velocity_from_target` and `.score_from_target`.
        """

        # Define the per-modality conversion as a function
        def _per_modality_network_output_to_generator(modality, net_out_k, z_t_k):
            # Use the conversion methods!
            velocity = modality.interpolant.velocity_from_target(net_out_k, z_t_k, t)
            score = modality.interpolant.score_from_target(net_out_k, z_t_k, t)
            return VelocityAndScore(velocity=velocity, score=score)

        # Use `.map` to apply the function over the whole pytree
        return self.modality_registry.map(
            _per_modality_network_output_to_generator, net_out, z_t
        )

In [ ]:
converter_denoiser_gen_model = ConverterDenoiserGenerativeModel(
    network, modality_registry
)

Let's now check to see whether the two implemenations align!

To do so we need a network output, `net_out`. Producing it mirrors what [`LossPipeline`](https://instadeepai.github.io/stix/api_reference/training/loss_pipeline.html#loss-pipeline) does internally each step: embed the batch, sample a time `t` and noise `epsilon`, interpolate to the noisy state `z_t`, then call the network. The only difference is we run on the full batch directly rather than per-sample under `vmap`.

In [ ]:
batch = next(train_iter)

# `get_network_output` is a thin wrapper over the network, so we must hand it the
# same inputs the `LossPipeline` assembles internally: the noisy state z_t and the
# time t. There is no conditioning in this tutorial, so context/attention masks are None.

# 1. Embed the raw batch -> per-modality (source, target) pairs in latent space.
embedded_pairs = denoiser_gen_model.get_embeddings(batch.raw_batch)

# 2. Sample a shared interpolation time and per-modality noise (split per modality).
key, t_key, noise_key = jr.split(key, 3)  # time + noise draws
t = jr.uniform(t_key)
epsilon = modality_registry.map(
    lambda modality, pair_k, key_k: modality.interpolant.sample_noise(
        key_k, pair_k.target.shape
    ),
    embedded_pairs,
    modality_registry.split_and_project_key(noise_key),
)

# 3. Form the noisy interpolated state z_t = beta_t z_{\mathrm{tgt}} + gamma_t epsilon.
z_t = modality_registry.map(
    lambda modality, pair_k, eps_k: modality.interpolant.interpolate(pair_k, t, eps_k),
    embedded_pairs,
    epsilon,
)

# 4. Call the network (no conditioning -> context/masks are None).
net_out = denoiser_gen_model.get_network_output(
    z_t, t, context_data=None, context_mask=None, attention_mask=None
)

In [ ]:
# The hand-derived `get_generator` and the interpolant's pre-baked
# `velocity_from_target` and `score_from_target` conversions are the same maths,
# so the two models must agree modality-by-modality.
manual_gen = denoiser_gen_model.get_generator(net_out, z_t, t)
prebaked_gen = converter_denoiser_gen_model.get_generator(net_out, z_t, t)

for modality_key in manual_gen:
    match = jnp.allclose(
        manual_gen[modality_key].velocity, prebaked_gen[modality_key].velocity
    )
    print(f"velocity | {modality_key:<12} match: {bool(match)}")

    match = jnp.allclose(
        manual_gen[modality_key].score, prebaked_gen[modality_key].score
    )
    print(f"score | {modality_key:<12} match: {bool(match)}")

## 3.2 Noise-prediction model

The `DenoiserGenerativeModel` trained the network to predict the clean data $z_{\mathrm{tgt}}$. A common alternative is to predict the **noise** $\epsilon$ instead (the classic "$\epsilon$-prediction" of diffusion models). The interpolant is unchanged, $z_t = t z_{\mathrm{tgt}} + (1-t) \epsilon$ — only the *target* changes, and that choice cascades into all three conversions.

Step 1:
**Loss**: An MSE between the network output and the noise $\epsilon$ drawn for the interpolation path (it arrives in `get_loss` as the `epsilon` argument).

Step 2:
**Velocity**: $v_t = \dot{z}_t = z_{\mathrm{tgt}} - \epsilon$. Substituting $z_{\mathrm{tgt}} = \frac{z_t - (1-t) \epsilon}{t}$ gives $v_t = \frac{z_t - \epsilon}{t}$.

Step 3:
**Score**: $s_t = -\frac{\epsilon}{\gamma_t} = -\frac{\epsilon}{1-t}$.

This is exactly the recipe of the predefined [`NoiseOneSidedGenerativeModel`](https://instadeepai.github.io/stix/api_reference/core/gen_model.html#stix.core.gen_model.factory.NoiseOneSidedGenerativeModel); we spell it out by hand here to contrast with the denoiser.

In [ ]:
class NoisePredictionGenerativeModel(GenerativeModel):
    """Noise predicting generative model."""

    def get_loss(self, net_out, t, z_t, epsilon, embedded_pairs, raw_pairs, loss_mask):
        """Compute the loss. An MSE between the sampled noise epsilon and our network output."""
        # Use one of our pre-defined criterion
        mse = MSECriterion()

        # Define the per-modality loss as a function
        def _per_modality_network_output_to_loss(
            modality, net_out_k, epsilon_k, loss_mask_k
        ):
            return mse(net_out_k, epsilon_k, t, loss_mask_k)

        # Use `.map` to apply the function over the whole pytree
        per_modality_loss = self.modality_registry.map(
            _per_modality_network_output_to_loss, epsilon, loss_mask
        )

        # Reduce the pytree to a scalar by computing the mean
        return jnp.mean(jnp.stack(jax.tree.leaves(per_modality_loss)))

    def get_generator(self, net_out, z_t, t):
        r"""Convert network output to a per-modality `VelocityAndScore` generator.

        We have $v_t = \frac{z_t - \epsilon}{t}$ and $s_t = -\frac{\epsilon}{1-t}$.
        The maths is derived in the markdown above!
        """

        # Define the per-modality conversion as a function
        def _per_modality_network_output_to_generator(modality, net_out_k, z_t_k):
            velocity = (z_t_k - net_out_k) / t
            score = -net_out_k / (1 - t)
            return VelocityAndScore(velocity=velocity, score=score)

        # Use `.map` to apply the function over the whole pytree
        return self.modality_registry.map(
            _per_modality_network_output_to_generator, net_out, z_t
        )

In [ ]:
noise_gen_model = NoisePredictionGenerativeModel(network, modality_registry)

## 3.3 (**Advanced**) Hybrid velocity + logits model

So far every model has treated all modalities (`coordinates` and `index`) *identically*. But the per-modality `.map` callback receives the `modality` itself, so we can easily customise thing such that **each modality can predict a different thing**. We will give an example now where we simply branch on `modality.is_discrete` inside each conversion, giving discrete and continuous modalities different treatments.

Here we build an hybrid model that directly predicts the velocity for the continuous modalities, while predicting the logits for the posterior distribution of the discrete indices. More precisely, the model will approximate the posterior distribution as $p(z_{\mathrm{tgt}}|z_t) \simeq \text{softmax}(\varphi(z_t))$, with $\varphi(z_t)$ the network output. As we are considering a linear interpolant, the velocity and score can be deduced from the knowledge of the estimated target $\hat{z}_{\mathrm{tgt}} = \mathbb{E}\left[z_{\mathrm{tgt}} | z_t\right]$. Because `index` uses a `OneHotDiscreteEmbedder`, and since we have an estimate for $p(z_{\mathrm{tgt}}|z_t)$, the estimated target can be computed as $\hat{z}_{\mathrm{tgt}} = \sum_{k=1}^{K} p(z_{\mathrm{tgt}} = k| z_t)e_k$, where $e_k$ is the one-hot vector for the categorical index $k$.

In a nutshell:

| | `coordinates` (continuous) | `index` (discrete) |
|---|---|---|
| network head predicts | **velocity** $v_t$ | **logits** |
| loss | MSE vs conditional velocity | cross-entropy vs one-hot label |
| `get_generator` (velocity) | identity | $\hat{z}_{\mathrm{tgt}} = \mathrm{softmax}(\text{logits})$, then `velocity_from_target` |
| `get_generator` (score) | `score_from_velocity` | $\hat{z}_{\mathrm{tgt}}$ as above, then `score_from_target` |

<br>

> **Note**:  Because MSE and cross-entropy live on different scales a real training may want to weight their loss contributions accordingly. Reducing the per-modality losses by hand amounts to `jnp.mean(jnp.stack(jax.tree.leaves(per_modality_loss)))`. `stix` provides the helper function `reduce_modality_losses`, which performs the same reduction and optionally takes a pytree of per-modality weights. We demonstrate how this can be used below. 

In [ ]:
# One weight for continuous datamodes, another for discrete.
# Upweight the discrete modalities by 5.
loss_weights = modality_registry.map(
    lambda modality: 5 if modality.is_discrete else 1.0
)


class HybridVelocityLogitsGenerativeModel(GenerativeModel):
    """Hybrid velocity (continuous modalities) and logits (discrete) generative model."""

    def get_loss(self, net_out, t, z_t, epsilon, embedded_pairs, raw_pairs, loss_mask):
        """Mixed loss: velocity MSE for continuous modalities, cross-entropy for discrete ones."""
        # A criterion for each kind of modality.
        mse = MSECriterion()
        cross_entropy = CrossEntropyCriterion()

        # Branch on `modality.is_discrete` to pick the right target and criterion.
        def _per_modality_network_output_to_loss(
            modality, net_out_k, embedded_pairs_k, raw_pairs_k, epsilon_k, loss_mask_k
        ):
            # Discrete
            if modality.is_discrete:
                # cross_entropy takes logits as input, so net_out_k is interpreted as logits
                return cross_entropy(net_out_k, raw_pairs_k.target, t, loss_mask_k)

            # Continuous
            conditional_velocity = modality.interpolant.get_conditional_velocity(
                embedded_pairs_k, t, epsilon_k
            )
            return mse(net_out_k, conditional_velocity, t, loss_mask_k)

        # Use `.map` to apply the function over the whole pytree
        per_modality_loss = self.modality_registry.map(
            _per_modality_network_output_to_loss,
            net_out,
            embedded_pairs,
            raw_pairs,
            epsilon,
            loss_mask,
        )

        # Reduce the pytree to a scalar by computing the mean
        return reduce_modality_losses(per_modality_loss, weights=loss_weights)

    def get_generator(self, net_out, z_t, t):
        r"""Convert network output to a per-modality `VelocityAndScore` generator.

        Continuous: the output already *is* the velocity (identity); the score is
        derived from it via `score_from_velocity`.
        Discrete: the flow lives in embedding space. `index` uses a one-hot
        embedder, so the embedding space is the simplex and the expected embedding
        under the predicted categorical is just the distribution itself:
        z1_hat = softmax(logits). Then `velocity_from_target` / `score_from_target`.
        """

        def _per_modality_network_output_to_generator(modality, net_out_k, z_t_k):
            # Discrete
            if modality.is_discrete:
                z1_hat = jax.nn.softmax(net_out_k, axis=-1)
                velocity = modality.interpolant.velocity_from_target(z1_hat, z_t_k, t)
                score = modality.interpolant.score_from_target(z1_hat, z_t_k, t)
                return VelocityAndScore(velocity=velocity, score=score)
            # Continuous
            velocity = net_out_k
            score = modality.interpolant.score_from_velocity(net_out_k, z_t_k, t)
            return VelocityAndScore(velocity=velocity, score=score)

        return self.modality_registry.map(
            _per_modality_network_output_to_generator, net_out, z_t
        )

In [ ]:
hybrid_gen_model = HybridVelocityLogitsGenerativeModel(network, modality_registry)

# The two criteria run side by side and reduce to a single scalar. `get_loss` needs
# the raw one-hot label (for cross-entropy) and a loss mask — all-ones here for the demo.
loss_mask = modality_registry.map(
    lambda modality, net_out_k: jnp.ones_like(net_out_k), net_out
)
loss = hybrid_gen_model.get_loss(
    net_out, t, z_t, epsilon, embedded_pairs, batch.raw_batch, loss_mask
)
print(f"hybrid loss (velocity MSE + cross-entropy): {loss:.4f}")

# And every modality still yields a velocity/score generator, whatever its target was.
generators = hybrid_gen_model.get_generator(net_out, z_t, t)
for modality_key, g in generators.items():
    print(f"velocity | {modality_key:<12} shape: {g.velocity.shape}")